# Optuna with Scikit‑Learn Pipelines — Regression

This notebook demonstrates end‑to‑end hyperparameter optimization using **Optuna** with a reusable scikit‑learn pipeline.

### Task
- **Dataset:** Palmer Penguins (via `seaborn`)
- **Target:** `body_mass_g` (penguin body weight in grams)
- **Model:** `GradientBoostingRegressor` with `StandardScaler`
- **Metric:** Negative RMSE (minimized → lower RMSE is better)

### Where Optuna Fits

- Dataset arrives, customer wants a model  
  - You perform best model and scaler selection
  - Once model and scaler identified:
    - Run Optuna to identify best hyperparameters for your model  

In [1]:
#!pip install optuna seaborn

## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer, make_column_selector as selector
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial noise

## 2. Load Sample Data

The Palmer Penguins dataset contains measurements for 344 penguins across three species.
We drop rows where `body_mass_g` (our target) is missing.

In [3]:
df = sns.load_dataset("penguins").dropna(subset=["body_mass_g"])

X = df.drop(columns="body_mass_g")
y = df["body_mass_g"]

print(f"Dataset shape: {X.shape}")
print(f"\nFeature dtypes:\n{X.dtypes}")
print(f"\nTarget summary:\n{y.describe()}")

Dataset shape: (342, 6)

Feature dtypes:
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
sex                   object
dtype: object

Target summary:
count     342.000000
mean     4201.754386
std       801.954536
min      2700.000000
25%      3550.000000
50%      4050.000000
75%      4750.000000
max      6300.000000
Name: body_mass_g, dtype: float64


## 3. Preprocessing

The penguins dataset contains missing values in both numeric and categorical columns.
`GradientBoostingRegressor` does not accept NaNs natively, so we impute before scaling/encoding.

- **Categorical columns** (`species`, `island`, `sex`): most-frequent imputation → `OrdinalEncoder`
- **Numeric columns** (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`): median imputation → `StandardScaler`

In [4]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
)

numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

preprocessor = make_column_transformer(
    (categorical_pipeline, selector(dtype_include=object)),   # species, island, sex
    (numeric_pipeline,     selector(dtype_include=np.number)), # bill/flipper measurements
    remainder="drop",
)

print("Preprocessor configured:")
print("  - most_frequent imputer + OrdinalEncoder → categorical columns")
print("  - median imputer + StandardScaler        → numeric columns")

Preprocessor configured:
  - most_frequent imputer + OrdinalEncoder → categorical columns
  - median imputer + StandardScaler        → numeric columns


## 4. Baseline Pipeline

Build the pipeline with default `GradientBoostingRegressor` settings first, so we have
a reference score before running Optuna.

In [5]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(random_state=42)),
])

# Baseline cross-validated RMSE
baseline_scores = cross_val_score(
    model, X, y,
    cv=5,
    scoring="neg_root_mean_squared_error",
)
print(f"Baseline CV RMSE: {-baseline_scores.mean():.1f} g  (±{baseline_scores.std():.1f} g)")

Baseline CV RMSE: 314.1 g  (±36.6 g)


## 5. Optuna Objective Function

We tune **7 hyperparameters** of `GradientBoostingRegressor`:

| Parameter | Type | Range / Choices | Effect |
|---|---|---|---|
| `n_estimators` | int | 50 – 500 | Number of boosting stages |
| `learning_rate` | float (log) | 1e-3 – 0.5 | Shrinks each tree's contribution |
| `max_depth` | int | 2 – 8 | Depth of individual trees |
| `min_samples_split` | int | 2 – 20 | Min samples to split an internal node |
| `min_samples_leaf` | int | 1 – 20 | Min samples at a leaf node |
| `subsample` | float | 0.5 – 1.0 | Fraction of samples per tree (stochastic boosting) |
| `max_features` | categorical | `sqrt`, `log2`, `None` | Features considered at each split |

Optuna **minimizes** the objective, so we return a positive RMSE.

In [6]:
def objective(trial):
    params = {
        "regressor__n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "regressor__learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),
        "regressor__max_depth": trial.suggest_int("max_depth", 2, 8),
        "regressor__min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "regressor__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "regressor__subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "regressor__max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }

    model_trial = clone(model)
    model_trial.set_params(**params)

    scores = cross_val_score(
        model_trial,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error",

    
    )

    # Return positive RMSE — Optuna minimizes
    return -scores.mean()

## 6. Run Optuna Study

In [7]:
%%time
study = optuna.create_study(direction="minimize")  # minimize RMSE
study.optimize(objective, n_trials=50)

print(f"\nBest RMSE  : {study.best_value:.1f} g")
print(f"Best params: {study.best_params}")


Best RMSE  : 302.1 g
Best params: {'n_estimators': 428, 'learning_rate': 0.08907618503853615, 'max_depth': 7, 'min_samples_split': 4, 'min_samples_leaf': 18, 'subsample': 0.5287423149049031, 'max_features': 'sqrt'}
CPU times: user 1min 12s, sys: 427 ms, total: 1min 13s
Wall time: 1min 15s


## 7. Train Final Model

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

best_model = clone(model)
best_model.set_params(**{
    f"regressor__{k}": v for k, v in study.best_params.items()
})

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test RMSE  : {test_rmse:.1f} g")
print(f"Test R²    : {best_model.score(X_test, y_test):.4f}")
print()
study.best_params

Test RMSE  : 320.9 g
Test R²    : 0.8451



{'n_estimators': 428,
 'learning_rate': 0.08907618503853615,
 'max_depth': 7,
 'min_samples_split': 4,
 'min_samples_leaf': 18,
 'subsample': 0.5287423149049031,
 'max_features': 'sqrt'}

In [9]:
best_model

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
